In [1]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [2]:
df = pd.read_csv("../datas/all_sites_combined.csv")

In [3]:
df.head()

,timestamp,site_id,site_type,site_name,consumption_kwh,consumption_euros,temperature_celsius,humidity_percent,solar_irradiance_wm2,hour,day_of_week,day_name,month,is_weekend,is_working_hours
0,2023-01-01 00:00:00,SITE001,office,Bureau Tertiaire,85.82,12.87,10.6,76.7,0.0,0,6,Sunday,1,1,0
1,2023-01-01 01:00:00,SITE001,office,Bureau Tertiaire,89.84,13.48,12.0,70.6,0.0,1,6,Sunday,1,1,0
2,2023-01-01 02:00:00,SITE001,office,Bureau Tertiaire,80.62,12.09,13.8,62.2,0.0,2,6,Sunday,1,1,0
3,2023-01-01 03:00:00,SITE001,office,Bureau Tertiaire,92.67,13.90,9.4,66.0,0.0,3,6,Sunday,1,1,0
4,2023-01-01 04:00:00,SITE001,office,Bureau Tertiaire,72.50,10.88,14.8,67.2,41.5,4,6,Sunday,1,1,0


In [4]:
df.shape

(122647, 15)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 122647 entries, 0 to 122646
Data columns (total 15 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   timestamp             122647 non-null  str    
 1   site_id               122647 non-null  str    
 2   site_type             122647 non-null  str    
 3   site_name             122647 non-null  str    
 4   consumption_kwh       119807 non-null  float64
 5   consumption_euros     120160 non-null  float64
 6   temperature_celsius   119231 non-null  float64
 7   humidity_percent      119224 non-null  float64
 8   solar_irradiance_wm2  118683 non-null  float64
 9   hour                  122647 non-null  int64  
 10  day_of_week           122647 non-null  int64  
 11  day_name              122647 non-null  str    
 12  month                 122647 non-null  int64  
 13  is_weekend            122647 non-null  int64  
 14  is_working_hours      122647 non-null  int64  
dtypes: float64(

In [6]:
df.isna().sum()

timestamp                  0
site_id                    0
site_type                  0
site_name                  0
consumption_kwh         2840
consumption_euros       2487
temperature_celsius     3416
humidity_percent        3423
solar_irradiance_wm2    3964
hour                       0
day_of_week                0
day_name                   0
month                      0
is_weekend                 0
is_working_hours           0
dtype: int64

In [7]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

df["timestamp"].min(), df["timestamp"].max()

(Timestamp('2023-01-01 00:00:00'), Timestamp('2024-12-31 00:00:00'))

In [8]:
sites_summary = (
    df.groupby(["site_id", "site_name", "site_type"])
    .agg(
        date_debut=("timestamp", "min"),
        date_fin=("timestamp", "max"),
        nombre_lignes=("timestamp", "size")
    )
    .reset_index()
)

sites_summary

,site_id,site_name,site_type,date_debut,date_fin,nombre_lignes
0,SITE001,Bureau Tertiaire,office,2023-01-01,2024-12-31,17521
1,SITE002,Usine de Production,factory,2023-01-01,2024-12-31,17521
2,SITE003,Data Center,datacenter,2023-01-01,2024-12-31,17521
3,SITE004,Centre Commercial,retail,2023-01-01,2024-12-31,17521
4,SITE005,Hôpital,hospital,2023-01-01,2024-12-31,17521
5,SITE006,Bureau Tertiaire,office,2023-01-01,2024-12-31,17521
6,SITE007,Usine de Production,factory,2023-01-01,2024-12-31,17521


In [9]:
# Copie de travail pour conserver le dataset brut intact
df_clean = df.copy()

# Tri chronologique par site
df_clean = df_clean.sort_values(
    ["site_id", "timestamp"]
).reset_index(drop=True)

In [10]:
df_clean.duplicated(
    subset=["site_id", "timestamp"]
).sum()

np.int64(0)

In [11]:
missing_by_site = (
    df_clean
    .groupby("site_id")[
        [
            "consumption_kwh",
            "temperature_celsius",
            "humidity_percent",
            "solar_irradiance_wm2"
        ]
    ]
    .apply(lambda x: x.isna().sum())
)

missing_by_site

,consumption_kwh,temperature_celsius,humidity_percent,solar_irradiance_wm2
site_id,,,,
SITE001,411,491,494,566
SITE002,407,491,489,569
SITE003,407,488,491,571
SITE004,405,485,487,576
SITE005,409,493,493,559
SITE006,398,491,489,558
SITE007,403,477,480,565


In [12]:
columns = [
    "site_id",
    "timestamp",
    "site_type",
    "consumption_kwh",
    "temperature_celsius",
    "humidity_percent",
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "is_working_hours"
]

df_clean = df[columns].copy()

In [13]:
df_clean.head()

,site_id,timestamp,site_type,consumption_kwh,temperature_celsius,humidity_percent,hour,day_of_week,month,is_weekend,is_working_hours
0,SITE001,2023-01-01 00:00:00,office,85.82,10.6,76.7,0,6,1,1,0
1,SITE001,2023-01-01 01:00:00,office,89.84,12.0,70.6,1,6,1,1,0
2,SITE001,2023-01-01 02:00:00,office,80.62,13.8,62.2,2,6,1,1,0
3,SITE001,2023-01-01 03:00:00,office,92.67,9.4,66.0,3,6,1,1,0
4,SITE001,2023-01-01 04:00:00,office,72.50,14.8,67.2,4,6,1,1,0


In [14]:
df.shape

(122647, 15)

In [15]:
# Conservation de la valeur brute
df_clean["consumption_kwh_corrected"] = df_clean["consumption_kwh"]
df_clean["temperature_celsius_corrected"] = df_clean["temperature_celsius"]
df_clean["humidity_percent_corrected"] = df_clean["humidity_percent"]

# Remplacement des valeurs manquantes avec la dernière valeur connue du même site
df_clean["consumption_kwh_corrected"] = (
    df_clean
    .groupby("site_id")["consumption_kwh_corrected"]
    .ffill()
)

df_clean["temperature_celsius_corrected"] = (
    df_clean
    .groupby("site_id")["temperature_celsius_corrected"]
    .ffill()
)

df_clean["humidity_percent_corrected"] = (
    df_clean
    .groupby("site_id")["humidity_percent_corrected"]
    .ffill()
)

# .ffill() = forward fill : quand une valeur est vide, Pandas reprend la dernière valeur connue juste avant.

In [16]:
df_clean[
    ["consumption_kwh_corrected","temperature_celsius_corrected","humidity_percent_corrected"]
].isna().sum()

consumption_kwh_corrected        0
temperature_celsius_corrected    0
humidity_percent_corrected       0
dtype: int64

In [17]:
df_clean.head()

,site_id,timestamp,site_type,consumption_kwh,temperature_celsius,humidity_percent,hour,day_of_week,month,is_weekend,is_working_hours,consumption_kwh_corrected,temperature_celsius_corrected,humidity_percent_corrected
0,SITE001,2023-01-01 00:00:00,office,85.82,10.6,76.7,0,6,1,1,0,85.82,10.6,76.7
1,SITE001,2023-01-01 01:00:00,office,89.84,12.0,70.6,1,6,1,1,0,89.84,12.0,70.6
2,SITE001,2023-01-01 02:00:00,office,80.62,13.8,62.2,2,6,1,1,0,80.62,13.8,62.2
3,SITE001,2023-01-01 03:00:00,office,92.67,9.4,66.0,3,6,1,1,0,92.67,9.4,66.0
4,SITE001,2023-01-01 04:00:00,office,72.50,14.8,67.2,4,6,1,1,0,72.50,14.8,67.2


In [18]:
# Tri chronologique
df_clean = df_clean.sort_values(["timestamp", "site_id"]).reset_index(drop=True)

# Dates de séparation
train_end = df_clean["timestamp"].quantile(0.70)
validation_end = df_clean["timestamp"].quantile(0.85)

train = df_clean[df_clean["timestamp"] <= train_end].copy()

validation = df_clean[
    (df_clean["timestamp"] > train_end) &
    (df_clean["timestamp"] <= validation_end)
].copy()

test = df_clean[df_clean["timestamp"] > validation_end].copy()

In [19]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 85855 entries, 0 to 85854
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   site_id                        85855 non-null  str           
 1   timestamp                      85855 non-null  datetime64[us]
 2   site_type                      85855 non-null  str           
 3   consumption_kwh                83886 non-null  float64       
 4   temperature_celsius            83355 non-null  float64       
 5   humidity_percent               83339 non-null  float64       
 6   hour                           85855 non-null  int64         
 7   day_of_week                    85855 non-null  int64         
 8   month                          85855 non-null  int64         
 9   is_weekend                     85855 non-null  int64         
 10  is_working_hours               85855 non-null  int64         
 11  consumption_kwh_corrected 

In [20]:
validation.info()

<class 'pandas.DataFrame'>
RangeIndex: 18396 entries, 85855 to 104250
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   site_id                        18396 non-null  str           
 1   timestamp                      18396 non-null  datetime64[us]
 2   site_type                      18396 non-null  str           
 3   consumption_kwh                17947 non-null  float64       
 4   temperature_celsius            17942 non-null  float64       
 5   humidity_percent               17945 non-null  float64       
 6   hour                           18396 non-null  int64         
 7   day_of_week                    18396 non-null  int64         
 8   month                          18396 non-null  int64         
 9   is_weekend                     18396 non-null  int64         
 10  is_working_hours               18396 non-null  int64         
 11  consumption_kwh_corre

In [21]:
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 18396 entries, 104251 to 122646
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   site_id                        18396 non-null  str           
 1   timestamp                      18396 non-null  datetime64[us]
 2   site_type                      18396 non-null  str           
 3   consumption_kwh                17974 non-null  float64       
 4   temperature_celsius            17934 non-null  float64       
 5   humidity_percent               17940 non-null  float64       
 6   hour                           18396 non-null  int64         
 7   day_of_week                    18396 non-null  int64         
 8   month                          18396 non-null  int64         
 9   is_weekend                     18396 non-null  int64         
 10  is_working_hours               18396 non-null  int64         
 11  consumption_kwh_corr

In [22]:
print("Train :", train["timestamp"].min(), "→", train["timestamp"].max())
print("Validation :", validation["timestamp"].min(), "→", validation["timestamp"].max())
print("Test :", test["timestamp"].min(), "→", test["timestamp"].max())

Train : 2023-01-01 00:00:00 → 2024-05-26 00:00:00
Validation : 2024-05-26 01:00:00 → 2024-09-12 12:00:00
Test : 2024-09-12 13:00:00 → 2024-12-31 00:00:00


In [23]:
# Entrainement Prophet


# Ici on prend les valeurs d'origine sans les null
prophet_train = (
    train[train["site_id"] == "SITE001"]
    [["timestamp", "consumption_kwh"]]
    .rename(columns={
        "timestamp": "ds",
        "consumption_kwh": "y"
    })
    .copy()
)

# Ici on prend les valeurs d'origine sans les null
prophet_validation = (
    validation[validation["site_id"] == "SITE001"]
    [["timestamp", "consumption_kwh"]]
    .dropna()
    .rename(columns={
        "timestamp": "ds",
        "consumption_kwh": "y"
    })
    .copy()
)

# Ici on prend les valeurs d'origine sans les null
prophet_test = (
    test[test["site_id"] == "SITE001"]
    [["timestamp", "consumption_kwh"]]
    .dropna()
    .rename(columns={
        "timestamp": "ds",
        "consumption_kwh": "y"
    })
    .copy()
)

In [24]:
from prophet import Prophet
prophet_model = Prophet()

prophet_model.fit(prophet_train)

16:12:20 - cmdstanpy - INFO - Chain [1] start processing
16:12:23 - cmdstanpy - INFO - Chain [1] done processing


In [25]:
validation_future = prophet_validation[["ds"]].copy()

prophet_validation_forecast = prophet_model.predict(validation_future)

In [26]:
prophet_validation_forecast.head()

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,additive_terms,additive_terms_lower,additive_terms_upper,daily,daily_lower,daily_upper,weekly,weekly_lower,weekly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat
0,2024-05-26 07:00:00,126.061561,0.058921,114.784301,126.061561,126.061561,-70.419615,-70.419615,-70.419615,-34.596747,-34.596747,-34.596747,-35.822868,-35.822868,-35.822868,0.0,0.0,0.0,55.641946
1,2024-05-26 08:00:00,126.071134,22.828669,138.387463,126.071134,126.071134,-50.167806,-50.167806,-50.167806,-14.883007,-14.883007,-14.883007,-35.284799,-35.284799,-35.284799,0.0,0.0,0.0,75.903328
2,2024-05-26 09:00:00,126.080708,47.182659,170.116321,126.080708,126.080708,-19.150273,-19.150273,-19.150273,15.428602,15.428602,15.428602,-34.578875,-34.578875,-34.578875,0.0,0.0,0.0,106.930434
3,2024-05-26 10:00:00,126.090281,78.742432,197.006890,126.090281,126.090281,8.610502,8.610502,8.610502,42.321688,42.321688,42.321688,-33.711186,-33.711186,-33.711186,0.0,0.0,0.0,134.700783
4,2024-05-26 11:00:00,126.099854,85.664317,202.505287,126.099854,126.099854,21.392051,21.392051,21.392051,54.081220,54.081220,54.081220,-32.689169,-32.689169,-32.689169,0.0,0.0,0.0,147.491906


In [27]:
prophet_results = prophet_validation[["ds", "y"]].copy()

prophet_results["yhat"] = prophet_validation_forecast["yhat"].values

prophet_results.head()

,ds,y,yhat
85897,2024-05-26 07:00:00,89.15,55.641946
85904,2024-05-26 08:00:00,89.37,75.903328
85911,2024-05-26 09:00:00,87.27,106.930434
85918,2024-05-26 10:00:00,83.89,134.700783
85925,2024-05-26 11:00:00,86.49,147.491906


In [28]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(
    prophet_results["y"],
    prophet_results["yhat"]
)

rmse = np.sqrt(
    mean_squared_error(
        prophet_results["y"],
        prophet_results["yhat"]
    )
)

print("MAE :", mae)
print("RMSE :", rmse)

MAE : 41.45877873097996
RMSE : 52.03519183689818


In [29]:
prophet_validation["y"].describe()

count    2533.000000
mean      106.016704
std        54.163418
min         0.000000
25%        71.730000
50%        82.980000
75%       145.330000
max       591.910000
Name: y, dtype: float64

In [30]:
# baseline naïve

site001 = (
    df_clean[df_clean["site_id"] == "SITE001"]
    .sort_values("timestamp")
    .copy()
)

site001["baseline_24h"] = (
    site001["consumption_kwh_corrected"]
    .shift(24)
)

In [31]:
site001.head()

,site_id,timestamp,site_type,consumption_kwh,temperature_celsius,humidity_percent,hour,day_of_week,month,is_weekend,is_working_hours,consumption_kwh_corrected,temperature_celsius_corrected,humidity_percent_corrected,baseline_24h
0,SITE001,2023-01-01 00:00:00,office,85.82,10.6,76.7,0,6,1,1,0,85.82,10.6,76.7,NaN
7,SITE001,2023-01-01 01:00:00,office,89.84,12.0,70.6,1,6,1,1,0,89.84,12.0,70.6,NaN
14,SITE001,2023-01-01 02:00:00,office,80.62,13.8,62.2,2,6,1,1,0,80.62,13.8,62.2,NaN
21,SITE001,2023-01-01 03:00:00,office,92.67,9.4,66.0,3,6,1,1,0,92.67,9.4,66.0,NaN
28,SITE001,2023-01-01 04:00:00,office,72.50,14.8,67.2,4,6,1,1,0,72.50,14.8,67.2,NaN


In [32]:
baseline_results = validation[
    validation["site_id"] == "SITE001"
][["timestamp", "consumption_kwh"]].copy()

baseline_results = baseline_results.dropna(subset=["consumption_kwh"])

baseline_results = baseline_results.merge(
    site001[["timestamp", "baseline_24h"]],
    on="timestamp",
    how="left"
)

baseline_results = baseline_results.dropna(subset=["baseline_24h"])

In [33]:
baseline_mae = mean_absolute_error(
    baseline_results["consumption_kwh"],
    baseline_results["baseline_24h"]
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        baseline_results["consumption_kwh"],
        baseline_results["baseline_24h"]
    )
)

print("Baseline MAE :", baseline_mae)
print("Baseline RMSE :", baseline_rmse)

Baseline MAE : 31.08040268456376
Baseline RMSE : 54.66346353712375


In [34]:
prophet_model_yearly = Prophet(
    yearly_seasonality=True
)

prophet_model_yearly.fit(prophet_train)

Yearly seasonality is enabled with less than 730 days (approximately 2 years) of history. The model may be under-identified, and the trend/seasonality decomposition can be unstable and dependent on the Prophet/Stan version. Consider disabling yearly seasonality or providing more history.
16:12:27 - cmdstanpy - INFO - Chain [1] start processing
16:12:28 - cmdstanpy - INFO - Chain [1] done processing


In [35]:
prophet_validation_forecast_yearly = prophet_model_yearly.predict(
    prophet_validation[["ds"]]
)

In [36]:
prophet_results_yearly = prophet_validation[["ds", "y"]].copy()

prophet_results_yearly["yhat"] = (
    prophet_validation_forecast_yearly["yhat"].values
)

In [37]:
mae_yearly = mean_absolute_error(
    prophet_results_yearly["y"],
    prophet_results_yearly["yhat"]
)

rmse_yearly = np.sqrt(
    mean_squared_error(
        prophet_results_yearly["y"],
        prophet_results_yearly["yhat"]
    )
)

print("MAE Prophet yearly :", mae_yearly)
print("RMSE Prophet yearly :", rmse_yearly)

MAE Prophet yearly : 29.64542247241631
RMSE Prophet yearly : 42.79979862048423


In [38]:
changepoint_values = [0.01, 0.05, 0.1, 0.5]

results = []

for value in changepoint_values:
    model = Prophet(
        daily_seasonality=True,
        weekly_seasonality=True,
        yearly_seasonality=True,
        changepoint_prior_scale=value
    )

    model.fit(prophet_train)

    forecast = model.predict(
        prophet_validation[["ds"]]
    )

    mae = mean_absolute_error(
        prophet_validation["y"],
        forecast["yhat"]
    )

    rmse = np.sqrt(
        mean_squared_error(
            prophet_validation["y"],
            forecast["yhat"]
        )
    )

    results.append({
        "changepoint_prior_scale": value,
        "MAE": mae,
        "RMSE": rmse
    })

Yearly seasonality is enabled with less than 730 days (approximately 2 years) of history. The model may be under-identified, and the trend/seasonality decomposition can be unstable and dependent on the Prophet/Stan version. Consider disabling yearly seasonality or providing more history.
16:12:30 - cmdstanpy - INFO - Chain [1] start processing
16:12:31 - cmdstanpy - INFO - Chain [1] done processing
Yearly seasonality is enabled with less than 730 days (approximately 2 years) of history. The model may be under-identified, and the trend/seasonality decomposition can be unstable and dependent on the Prophet/Stan version. Consider disabling yearly seasonality or providing more history.
16:12:33 - cmdstanpy - INFO - Chain [1] start processing
16:12:34 - cmdstanpy - INFO - Chain [1] done processing
Yearly seasonality is enabled with less than 730 days (approximately 2 years) of history. The model may be under-identified, and the trend/seasonality decomposition can be unstable and dependent o

In [39]:
pd.DataFrame(results)

,changepoint_prior_scale,MAE,RMSE
0,0.01,32.044555,44.705661
1,0.05,29.645422,42.799799
2,0.10,27.039761,41.326951
3,0.50,73.031345,88.794547


In [40]:
df_catboost = (
    df_clean
    .sort_values(["site_id", "timestamp"])
    .copy()
)

In [41]:
df_catboost["consumption_lag_1h"] = (
    df_catboost
    .groupby("site_id")["consumption_kwh_corrected"]
    .shift(1)
)

df_catboost["consumption_lag_2h"] = (
    df_catboost
    .groupby("site_id")["consumption_kwh_corrected"]
    .shift(2)
)

df_catboost["consumption_lag_24h"] = (
    df_catboost
    .groupby("site_id")["consumption_kwh_corrected"]
    .shift(24)
)

df_catboost["consumption_lag_48h"] = (
    df_catboost
    .groupby("site_id")["consumption_kwh_corrected"]
    .shift(48)
)

df_catboost["consumption_lag_168h"] = (
    df_catboost
    .groupby("site_id")["consumption_kwh_corrected"]
    .shift(168)
)

In [42]:
df_catboost["rolling_mean_24h"] = (
    df_catboost
    .groupby("site_id")["consumption_kwh_corrected"]
    .transform(
        lambda x: x.shift(1).rolling(24).mean()
    )
)

df_catboost["rolling_mean_168h"] = (
    df_catboost
    .groupby("site_id")["consumption_kwh_corrected"]
    .transform(
        lambda x: x.shift(1).rolling(168).mean()
    )
)

In [46]:
df_catboost.head()

,site_id,timestamp,site_type,consumption_kwh,temperature_celsius,humidity_percent,hour,day_of_week,month,is_weekend,...,consumption_kwh_corrected,temperature_celsius_corrected,humidity_percent_corrected,consumption_lag_1h,consumption_lag_2h,consumption_lag_24h,consumption_lag_48h,consumption_lag_168h,rolling_mean_24h,rolling_mean_168h
0,SITE001,2023-01-01 00:00:00,office,85.82,10.6,76.7,0,6,1,1,...,85.82,10.6,76.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,SITE001,2023-01-01 01:00:00,office,89.84,12.0,70.6,1,6,1,1,...,89.84,12.0,70.6,85.82,NaN,NaN,NaN,NaN,NaN,NaN
14,SITE001,2023-01-01 02:00:00,office,80.62,13.8,62.2,2,6,1,1,...,80.62,13.8,62.2,89.84,85.82,NaN,NaN,NaN,NaN,NaN
21,SITE001,2023-01-01 03:00:00,office,92.67,9.4,66.0,3,6,1,1,...,92.67,9.4,66.0,80.62,89.84,NaN,NaN,NaN,NaN,NaN
28,SITE001,2023-01-01 04:00:00,office,72.50,14.8,67.2,4,6,1,1,...,72.50,14.8,67.2,92.67,80.62,NaN,NaN,NaN,NaN,NaN


,site_id,site_type,hour,day_of_week,month,is_weekend,is_working_hours,consumption_lag_1h,consumption_lag_2h,consumption_lag_24h,consumption_lag_48h,consumption_lag_168h,rolling_mean_24h,rolling_mean_168h,consumption_kwh_corrected
0,SITE001,office,0,6,1,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,85.82
7,SITE001,office,1,6,1,1,0,85.82,NaN,NaN,NaN,NaN,NaN,NaN,89.84
14,SITE001,office,2,6,1,1,0,89.84,85.82,NaN,NaN,NaN,NaN,NaN,80.62
21,SITE001,office,3,6,1,1,0,80.62,89.84,NaN,NaN,NaN,NaN,NaN,92.67
28,SITE001,office,4,6,1,1,0,92.67,80.62,NaN,NaN,NaN,NaN,NaN,72.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1190,SITE001,office,2,6,1,1,0,93.12,87.76,88.86,64.12,80.62,89.262500,95.087024,79.75
1197,SITE001,office,3,6,1,1,0,79.75,93.12,83.57,64.12,92.67,88.882917,95.081845,84.89
1204,SITE001,office,4,6,1,1,0,84.89,79.75,71.33,67.46,72.50,88.937917,95.035536,76.40
1211,SITE001,office,5,6,1,1,0,76.40,84.89,87.53,59.96,30.20,89.149167,95.058750,78.06


In [47]:
catboost_train = df_catboost[
    df_catboost["timestamp"] <= train_end
].copy()

catboost_validation = df_catboost[
    (df_catboost["timestamp"] > train_end) &
    (df_catboost["timestamp"] <= validation_end)
].copy()

In [48]:
target_train = "consumption_kwh_corrected"

In [49]:
catboost_train = catboost_train.dropna(
    subset=features + [target_train]
)

In [50]:
X_train = catboost_train[features]
y_train = catboost_train[target_train]

In [51]:
catboost_validation_site001 = (
    catboost_validation[
        catboost_validation["site_id"] == "SITE001"
    ]
    .dropna(subset=features + ["consumption_kwh"])
    .copy()
)

X_validation = catboost_validation_site001[features]
y_validation = catboost_validation_site001["consumption_kwh"]

In [52]:
print("Train :", X_train.shape)
print("Validation SITE001 :", X_validation.shape)

print()
print(X_train.head())

Train : (84679, 14)
Validation SITE001 : (2533, 14)

      site_id site_type  hour  day_of_week  month  is_weekend  \
1176  SITE001    office     0            6      1           1   
1183  SITE001    office     1            6      1           1   
1190  SITE001    office     2            6      1           1   
1197  SITE001    office     3            6      1           1   
1204  SITE001    office     4            6      1           1   

      is_working_hours  consumption_lag_1h  consumption_lag_2h  \
1176                 0              119.41              117.82   
1183                 0               87.76              119.41   
1190                 0               93.12               87.76   
1197                 0               79.75               93.12   
1204                 0               84.89               79.75   

      consumption_lag_24h  consumption_lag_48h  consumption_lag_168h  \
1176                80.30                68.05                 85.82   
1183           

In [53]:
from catboost import CatBoostRegressor

catboost_model = CatBoostRegressor(
    loss_function="RMSE",
    random_seed=42,
    verbose=False
)

catboost_model.fit(
    X_train,
    y_train,
    cat_features=["site_id", "site_type"]
)

CatBoostRegressor(loss_function='RMSE', random_seed=42, verbose=False)

In [54]:
site001_history = (
    df_catboost[
        (df_catboost["site_id"] == "SITE001") &
        (df_catboost["timestamp"] <= train_end)
    ]
    .sort_values("timestamp")
    ["consumption_kwh_corrected"]
    .tolist()
)

In [55]:
site001_validation_recursive = (
    df_catboost[
        (df_catboost["site_id"] == "SITE001") &
        (df_catboost["timestamp"] > train_end) &
        (df_catboost["timestamp"] <= validation_end)
    ]
    .sort_values("timestamp")
    .copy()
)

In [56]:
print("Historique disponible :", len(site001_history))
print("Heures à prédire :", len(site001_validation_recursive))

Historique disponible : 12265
Heures à prédire : 2628


In [57]:
history = site001_history.copy()
recursive_predictions = []

for _, row in site001_validation_recursive.iterrows():

    X_row = pd.DataFrame([{
        "site_id": row["site_id"],
        "site_type": row["site_type"],
        "hour": row["hour"],
        "day_of_week": row["day_of_week"],
        "month": row["month"],
        "is_weekend": row["is_weekend"],
        "is_working_hours": row["is_working_hours"],

        "consumption_lag_1h": history[-1],
        "consumption_lag_2h": history[-2],
        "consumption_lag_24h": history[-24],
        "consumption_lag_48h": history[-48],
        "consumption_lag_168h": history[-168],

        "rolling_mean_24h": np.mean(history[-24:]),
        "rolling_mean_168h": np.mean(history[-168:])
    }], columns=features)

    prediction = catboost_model.predict(X_row)[0]

    recursive_predictions.append({
        "timestamp": row["timestamp"],
        "y": row["consumption_kwh"],
        "yhat": prediction
    })

    history.append(prediction)

In [58]:
catboost_results = pd.DataFrame(recursive_predictions)

catboost_results.head()

,timestamp,y,yhat
0,2024-05-26 01:00:00,NaN,79.664797
1,2024-05-26 02:00:00,NaN,77.711004
2,2024-05-26 03:00:00,NaN,76.674610
3,2024-05-26 04:00:00,NaN,77.100042
4,2024-05-26 05:00:00,NaN,77.302588


In [59]:
catboost_results["y"].isna().sum()

np.int64(95)

In [60]:
catboost_results_eval = (
    catboost_results
    .dropna(subset=["y"])
    .copy()
)

print(catboost_results_eval.shape)

(2533, 3)


In [61]:
catboost_mae = mean_absolute_error(
    catboost_results_eval["y"],
    catboost_results_eval["yhat"]
)

catboost_rmse = np.sqrt(
    mean_squared_error(
        catboost_results_eval["y"],
        catboost_results_eval["yhat"]
    )
)

print("CatBoost MAE :", catboost_mae)
print("CatBoost RMSE :", catboost_rmse)

CatBoost MAE : 16.390095998273438
CatBoost RMSE : 33.86918827413612


In [62]:
df_catboost["temperature_lag_1h"] = (
    df_catboost
    .groupby("site_id")["temperature_celsius_corrected"]
    .shift(1)
)

df_catboost["humidity_lag_1h"] = (
    df_catboost
    .groupby("site_id")["humidity_percent_corrected"]
    .shift(1)
)

In [63]:
features_weather = features + [
    "temperature_lag_1h",
    "humidity_lag_1h"
]

In [64]:
df_catboost[
    [
        "timestamp",
        "site_id",
        "temperature_celsius_corrected",
        "temperature_lag_1h",
        "humidity_percent_corrected",
        "humidity_lag_1h"
    ]
].head()

,timestamp,site_id,temperature_celsius_corrected,temperature_lag_1h,humidity_percent_corrected,humidity_lag_1h
0,2023-01-01 00:00:00,SITE001,10.6,NaN,76.7,NaN
7,2023-01-01 01:00:00,SITE001,12.0,10.6,70.6,76.7
14,2023-01-01 02:00:00,SITE001,13.8,12.0,62.2,70.6
21,2023-01-01 03:00:00,SITE001,9.4,13.8,66.0,62.2
28,2023-01-01 04:00:00,SITE001,14.8,9.4,67.2,66.0


In [65]:
catboost_train_weather = df_catboost[
    df_catboost["timestamp"] <= train_end
].copy()

catboost_train_weather = catboost_train_weather.dropna(
    subset=features_weather + ["consumption_kwh_corrected"]
)

X_train_weather = catboost_train_weather[features_weather]
y_train_weather = catboost_train_weather["consumption_kwh_corrected"]

In [68]:
site001_validation_recursive = (
    df_catboost[
        (df_catboost["site_id"] == "SITE001") &
        (df_catboost["timestamp"] > train_end) &
        (df_catboost["timestamp"] <= validation_end)
    ]
    .sort_values("timestamp")
    .copy()
)

In [69]:
from catboost import CatBoostRegressor

catboost_model_weather = CatBoostRegressor(
    loss_function="RMSE",
    random_seed=42,
    verbose=False
)

catboost_model_weather.fit(
    X_train_weather,
    y_train_weather,
    cat_features=["site_id", "site_type"]
)

CatBoostRegressor(loss_function='RMSE', random_seed=42, verbose=False)

In [70]:
history = site001_history.copy()
recursive_predictions_weather = []

for _, row in site001_validation_recursive.iterrows():

    X_row = pd.DataFrame([{
        "site_id": row["site_id"],
        "site_type": row["site_type"],
        "hour": row["hour"],
        "day_of_week": row["day_of_week"],
        "month": row["month"],
        "is_weekend": row["is_weekend"],
        "is_working_hours": row["is_working_hours"],

        "consumption_lag_1h": history[-1],
        "consumption_lag_2h": history[-2],
        "consumption_lag_24h": history[-24],
        "consumption_lag_48h": history[-48],
        "consumption_lag_168h": history[-168],

        "rolling_mean_24h": np.mean(history[-24:]),
        "rolling_mean_168h": np.mean(history[-168:]),

        "temperature_lag_1h": row["temperature_lag_1h"],
        "humidity_lag_1h": row["humidity_lag_1h"]

    }], columns=features_weather)

    prediction = catboost_model_weather.predict(X_row)[0]

    recursive_predictions_weather.append({
        "timestamp": row["timestamp"],
        "y": row["consumption_kwh"],
        "yhat": prediction
    })

    history.append(prediction)

In [71]:
catboost_results_weather = pd.DataFrame(
    recursive_predictions_weather
)

catboost_results_weather.head()

,timestamp,y,yhat
0,2024-05-26 01:00:00,NaN,79.462826
1,2024-05-26 02:00:00,NaN,79.716477
2,2024-05-26 03:00:00,NaN,79.666445
3,2024-05-26 04:00:00,NaN,81.121112
4,2024-05-26 05:00:00,NaN,82.405274


In [72]:
catboost_results_weather_eval = (
    catboost_results_weather
    .dropna(subset=["y"])
    .copy()
)

catboost_weather_mae = mean_absolute_error(
    catboost_results_weather_eval["y"],
    catboost_results_weather_eval["yhat"]
)

catboost_weather_rmse = np.sqrt(
    mean_squared_error(
        catboost_results_weather_eval["y"],
        catboost_results_weather_eval["yhat"]
    )
)

print("CatBoost + météo MAE :", catboost_weather_mae)
print("CatBoost + météo RMSE :", catboost_weather_rmse)

CatBoost + météo MAE : 17.79928852246051
CatBoost + météo RMSE : 34.366109412129205


In [73]:
def evaluate_catboost_recursive(model):
    history = site001_history.copy()
    predictions = []

    for _, row in site001_validation_recursive.iterrows():

        X_row = pd.DataFrame([{
            "site_id": row["site_id"],
            "site_type": row["site_type"],
            "hour": row["hour"],
            "day_of_week": row["day_of_week"],
            "month": row["month"],
            "is_weekend": row["is_weekend"],
            "is_working_hours": row["is_working_hours"],

            "consumption_lag_1h": history[-1],
            "consumption_lag_2h": history[-2],
            "consumption_lag_24h": history[-24],
            "consumption_lag_48h": history[-48],
            "consumption_lag_168h": history[-168],

            "rolling_mean_24h": np.mean(history[-24:]),
            "rolling_mean_168h": np.mean(history[-168:])
        }], columns=features)

        prediction = model.predict(X_row)[0]

        predictions.append({
            "y": row["consumption_kwh"],
            "yhat": prediction
        })

        history.append(prediction)

    results = pd.DataFrame(predictions).dropna(subset=["y"])

    mae = mean_absolute_error(
        results["y"],
        results["yhat"]
    )

    rmse = np.sqrt(
        mean_squared_error(
            results["y"],
            results["yhat"]
        )
    )

    return mae, rmse

In [74]:
parameter_sets = [
    {"iterations": 500,  "depth": 6,  "learning_rate": 0.03},
    {"iterations": 500,  "depth": 6,  "learning_rate": 0.10},
    {"iterations": 500,  "depth": 8,  "learning_rate": 0.03},
    {"iterations": 500,  "depth": 8,  "learning_rate": 0.10},
    {"iterations": 1000, "depth": 6,  "learning_rate": 0.03},
    {"iterations": 1000, "depth": 6,  "learning_rate": 0.10},
    {"iterations": 1000, "depth": 8,  "learning_rate": 0.03},
    {"iterations": 1000, "depth": 8,  "learning_rate": 0.10}
]

In [75]:
tuning_results = []

for params in parameter_sets:

    model = CatBoostRegressor(
        iterations=params["iterations"],
        depth=params["depth"],
        learning_rate=params["learning_rate"],
        loss_function="RMSE",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_train,
        y_train,
        cat_features=["site_id", "site_type"]
    )

    mae, rmse = evaluate_catboost_recursive(model)

    tuning_results.append({
        **params,
        "MAE": mae,
        "RMSE": rmse
    })

In [76]:
tuning_df = pd.DataFrame(tuning_results)

tuning_df.sort_values("MAE")

,iterations,depth,learning_rate,MAE,RMSE
7,1000,8,0.10,14.492754,32.871083
3,500,8,0.10,15.275278,33.146486
5,1000,6,0.10,16.403024,33.498588
6,1000,8,0.03,16.870695,34.042661
4,1000,6,0.03,17.429242,34.313745
1,500,6,0.10,17.498215,34.201285
2,500,8,0.03,17.810931,34.546401
0,500,6,0.03,19.108842,35.214220
